# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [1]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [2]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [3]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[ 0.0554,  0.0203, -0.0775, -0.0243,  0.1416,  0.0288,  0.1500, -0.1662],
        [-0.0044,  0.0129, -0.0303, -0.0017,  0.0309,  0.0465,  0.0373, -0.0204]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [4]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cuda:0
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [5]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [6]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[-0.1051,  0.6040],
        [-0.6484, -0.4814],
        [-0.7587, -1.8292],
        [-0.4261, -0.4157],
        [ 0.2965, -0.8944]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[-0.4971,  0.2086],
        [-0.3966, -0.1758],
        [ 0.1048, -0.4380],
        [-0.4123, -0.1653],
        [ 0.3860, -0.3885]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [8]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)


Output no_grad method:
 tensor([[-0.2791, -0.1431],
        [ 0.5436,  0.3282],
        [-0.0093,  0.2179],
        [ 0.0541,  0.1379],
        [ 0.3095,  0.3396]])
Output inference_mode method:
 tensor([[-0.2791, -0.1431],
        [ 0.5436,  0.3282],
        [-0.0093,  0.2179],
        [ 0.0541,  0.1379],
        [ 0.3095,  0.3396]])
Output no_grad context manager:
 tensor([[-0.2791, -0.1431],
        [ 0.5436,  0.3282],
        [-0.0093,  0.2179],
        [ 0.0541,  0.1379],
        [ 0.3095,  0.3396]])
Output inference_mode context manager:
 tensor([[-0.2791, -0.1431],
        [ 0.5436,  0.3282],
        [-0.0093,  0.2179],
        [ 0.0541,  0.1379],
        [ 0.3095,  0.3396]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [11]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [12]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0.0000, 0.7440, 0.0000, 0.0000],
        [0.0000, 0.2831, 0.0000, 0.0000]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[1.0743, 0.3720, 0.0000, 0.0000],
        [0.0000, 0.1415, 0.0000, 0.0000]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [13]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [15]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [16]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])



## Работа на семинаре

### LSTM

![lstm](assets/LSTM.png)


In [57]:
from typing import Tuple

class LSTMCell(torch.nn.Module):
    def __init__(self, input_size: int, hidden_size: int, bias: bool=True) -> None:
        super().__init__()

        self.input_size = input_size # I
        self.hidden_size = hidden_size # H
        
        self.linear_fcio_input = torch.nn.Linear(input_size, 4*hidden_size, bias=bias)
        self.linear_fcio_hidden = torch.nn.Linear(hidden_size, 4*hidden_size, bias=bias)
        
        self.sigmoid = torch.nn.Sigmoid()
        self.tanh = torch.nn.Tanh()

    # буду возвращать два h_next, c_next
    def forward(self, 
                x: torch.Tensor, 
                h_prev: torch.Tensor, 
                c_prev: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:

        f_c_i_o = self.linear_fcio_input(x)  + self.linear_fcio_hidden(h_prev) 
        f, c, i, o = torch.split(f_c_i_o, self.hidden_size, dim=-1)
        f = self.sigmoid(f)
        c = self.tanh(c)
        i = self.sigmoid(i)
        o = self.sigmoid(o)
        c_next = f * c_prev + c * i 
        h_next = self.tanh(c_next) * o
        return h_next, c_next


lstm = LSTMCell(3, 4)
x = torch.rand((1, 3))
h = torch.rand((1, 4))
c = torch.rand((1, 4))
lstm(x, h, c)

(tensor([[0.0950, 0.0847, 0.0013, 0.0438]], grad_fn=<MulBackward0>),
 tensor([[0.2069, 0.1794, 0.0018, 0.0896]], grad_fn=<AddBackward0>))

### Inception

![inception](assets/inception.png)

In [69]:
class BlockConv2d(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int, 
                 kernel_size: int, padding:int=0) -> None:
        super().__init__()
        self.conv = torch.nn.Conv2d(in_channels, out_channels, 
                                    kernel_size=kernel_size, padding=padding,
                                    bias=False)
        self.bn = torch.nn.BatchNorm2d(out_channels)
        self.activation  = torch.nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = self.bn(x)
        return self.activation(x)
    
class InceptionModule(torch.nn.Module):
    def __init__(self, in_ch: int, 
                 out_ch1x1: int, 
                 in_ch3x3: int, out_ch3x3: int,
                 in_ch5x5: int, out_ch5x5: int,
                 out_pool_proj: int) -> None:
        
        super().__init__()

        self.branch1x1 = BlockConv2d(in_ch, out_ch1x1, 1)
        
        self.branch3x3 = torch.nn.Sequential(
            BlockConv2d(in_ch, in_ch3x3, 1),
            BlockConv2d(in_ch3x3, out_ch3x3, 3, padding=1)
        )
        self.branch5x5 = torch.nn.Sequential(
            BlockConv2d(in_ch, in_ch5x5, 1),
            BlockConv2d(in_ch5x5, out_ch5x5, 5, padding=2)
        )
        self.branch_pool = torch.nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            BlockConv2d(in_ch, out_pool_proj, 1),
        )


    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_1x1 = self.branch1x1(x)
        x_3x3 = self.branch3x3(x)
        x_5x5 = self.branch5x5(x)
        x_pool = self.branch_pool(x)
        return torch.concat([x_1x1, x_3x3, x_5x5, x_pool], dim=1) 

# block 3a из Table 1  https://arxiv.org/pdf/1409.4842
inception3a = InceptionModule(192, 64, 96, 128, 16, 32, 32) # выход 64 + 128 + 32 + 32 = 256
x = torch.rand((1, 192, 28, 28))

inception3a(x).shape

torch.Size([1, 256, 28, 28])

### SE

![se](assets/SqueezeAndExcite.png)

In [79]:
class SqueezeExcitationBlock(torch.nn.Module):
    def __init__(self, in_ch: int, out_ch: int, red_coef: int) -> None:
        super().__init__()
        # Я выбрал такую трансформацию
        self.tr = torch.nn.Sequential(
            torch.nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            torch.nn.BatchNorm2d(out_ch),
            torch.nn.ReLU()
        )
        self.sq = torch.nn.AdaptiveAvgPool2d((1, 1))
        self.ex = torch.nn.Sequential(
            torch.nn.Conv2d(out_ch, out_ch//red_coef, 1), # роль FC
            torch.nn.ReLU(inplace=True),
            torch.nn.Conv2d(out_ch//red_coef, out_ch, 1), # роль FC
            torch.nn.Sigmoid()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # х [B, C, H, W]
        x = self.tr(x)        
        scale_factor = self.ex(self.sq(x)) # [B, C, 1, 1]
        return scale_factor * x

x = torch.rand((1, 3, 64, 64))
sq_ex_block =  SqueezeExcitationBlock(3, 128, 4)
sq_ex_block(x).shape

torch.Size([1, 128, 64, 64])

### Selective Kernel

![selective](assets/SelectiveKernel.png)


In [89]:
# в статье https://arxiv.org/pdf/1903.06586 используюется примерно такой вариант
# я взял depthwise из (depthwise/grouped)
class DepthwiseSeparableConvolution(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 depth_multiplier: int, 
                 kernel_size: int, padding:int=0) -> None:
        super().__init__()         
        hidden_channels = depth_multiplier * in_channels
        self.depth = torch.nn.Conv2d(in_channels, hidden_channels, 
                                    kernel_size=kernel_size, padding=padding,
                                    groups=in_channels,
                                    bias=False)
        self.point = torch.nn.Conv2d(hidden_channels, out_channels, 
                                    kernel_size=1, bias=False)
        self.bn = torch.nn.BatchNorm2d(out_channels)
        self.activation = torch.nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depth(x)
        x = self.point(x)
        x = self.bn(x)
        return self.activation(x)

class SelectiveKernel(torch.nn.Module):
    def __init__(self, in_ch: int, out_ch: int, 
                 depth_multiplier: int,
                 red_coef: int) -> None:
        super().__init__()
        self.d = max(out_ch//red_coef, 32)
        self.branch3x3 = DepthwiseSeparableConvolution(in_ch, out_ch, depth_multiplier, 3, 1)
        self.branch5x5 = DepthwiseSeparableConvolution(in_ch, out_ch, depth_multiplier, 5, 2)
        self.global_avg_pool = torch.nn.AdaptiveAvgPool2d((1, 1))

        self.fc = torch.nn.Sequential(
            torch.nn.Conv2d(out_ch, self.d, 1, bias=False),
            torch.nn.BatchNorm2d(self.d),
            torch.nn.ReLU(inplace=True)
        )
        self.A = torch.nn.Conv2d(self.d, out_ch, 1)
        self.B = torch.nn.Conv2d(self.d, out_ch, 1)
        self.softmax = torch.nn.Softmax(dim=1)        
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x = [B, C, H, W]
        # split
        x_3x3 = self.branch3x3(x)
        x_5x5 = self.branch5x5(x)
        #fuse
        s = self.global_avg_pool(x_3x3 + x_5x5)  # [B, C, 1, 1]
        z = self.fc(s)                           # [B, D, 1, 1]  
        #select
        a = self.A(z)
        b = self.B(z)
        attn = torch.stack([a, b], dim=1)        # [B, 2, D, 1, 1]
        attn = self.softmax(attn)
        a_norm = attn[:, 0]                      # [B, D, 1, 1]
        b_norm = attn[:, 1]                      # [B, D, 1, 1]
        return a_norm * x_3x3 + b_norm * x_5x5

        
        
x = torch.rand(4, 128, 64, 64)
sk = SelectiveKernel(128, 256, 2, 16)
sk(x).shape

torch.Size([4, 256, 64, 64])

### PatchMerger

![patchmerger](assets/PatchMerger.png)


In [97]:
# https://arxiv.org/pdf/2202.12015

class PatchMerger(torch.nn.Module):
    def __init__(self, emb_dim: int, out_dim: int) -> None:
        super().__init__()
        self.ln = torch.nn.LayerNorm(emb_dim)
        self.learned_weights = torch.nn.Parameter(torch.randn(out_dim, emb_dim))  # [M, D]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.ln(x)                                    # [B, N, D]
        scores = x @ self.learned_weights.transpose(0, 1) # [B, N, M]
        attn = scores.softmax(dim=2).transpose(1, 2)      # [B, M, N]
        return attn @ x                                   # [B, M, D]

x = torch.rand(4, 16, 64) # [B, N, D]
pm = PatchMerger(64, 8)
pm(x).shape

torch.Size([4, 8, 64])

## Homework

2 задания:
1. Реализуйте требуемый в заголовке блок (максмсум 0.8 балов).

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [99]:
class ResidualBlock(torch.nn.Module):
    def __init__(self, in_ch: int) -> None:
        super().__init__()

        self.layers = torch.nn.Sequential(
            torch.nn.Conv2d(in_ch, in_ch, 3, padding=1, bias=False),
            torch.nn.BatchNorm2d(in_ch),
            torch.nn.ReLU(inplace=True),
            
            torch.nn.Conv2d(in_ch, in_ch, 3, padding=1, bias=False),
            torch.nn.BatchNorm2d(in_ch), 
        )
        self.relu = torch.nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.relu(x + self.layers(x))


x = torch.rand((4, 256, 64, 64))
res_block = ResidualBlock(256)
res_block(x).shape

torch.Size([4, 256, 64, 64])

## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [100]:
class SeparableConv2d(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int,
                 depth_multiplier: int, 
                 kernel_size: int, padding:int=0) -> None:
        super().__init__()         
        hidden_channels = depth_multiplier * in_channels
        self.depth = torch.nn.Conv2d(in_channels, hidden_channels, 
                                    kernel_size=kernel_size, padding=padding,
                                    groups=in_channels,
                                    bias=False)
        self.point = torch.nn.Conv2d(hidden_channels, out_channels, 
                                    kernel_size=1, bias=False)
        self.bn = torch.nn.BatchNorm2d(out_channels)
        self.activation = torch.nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depth(x)
        x = self.point(x)
        x = self.bn(x)
        return self.activation(x)
x = torch.rand(4, 32, 128, 128)    
sep_conv = SeparableConv2d(32, 64, 1, 3, 1)
sep_conv(x).shape

torch.Size([4, 64, 128, 128])

## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [101]:
from typing import Optional
import torch
from torch import nn
import numpy as np

class VanillaAttention(torch.nn.Module):
    def __init__(self, emb_dim: int) -> None:
        super().__init__()
        self.W_align = torch.nn.Linear(emb_dim, emb_dim, bias=False)
        self.W_value = torch.nn.Linear(emb_dim, emb_dim, bias=False)
        self.W_query = torch.nn.Linear(emb_dim, emb_dim, bias=False)
        self.softmax = torch.nn.Softmax(dim=1)
        self.tanh = torch.nn.Tanh()

    def forward(self, q: torch.Tensor, k: torch.Tensor) -> torch.Tensor:
        # q [B, d]
        # k [B, L, d]
        q_aligned = self.W_align(q)   # [B, d] @ [d, d] -> [B, d]
        score = k @ q_aligned.unsqueeze(2)       # [B, L, d] @ [B, d, 1] -> [B, L, 1]
        att = self.softmax(score.squeeze(2))     # [B, L]
        context = torch.sum(att.unsqueeze(2) * k, dim=1)  # [B, d]
        return self.tanh(self.W_value(context) + self.W_query(q))
q = torch.rand((4, 64))
k = torch.rand((4, 16, 64))
van_att = VanillaAttention(64)
van_att(q, k).shape

torch.Size([4, 64])

## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [103]:
# сделал без маски
class ScaledDotProductAttention(nn.Module):
    def __init__(self, emb_dim: int) -> None:
        super().__init__()
        self.register_buffer("scale", torch.tensor(emb_dim ** 0.5))
        self.softmax = torch.nn.Softmax(dim=-1)

    def forward(self, 
                q: torch.Tensor, 
                k: torch.Tensor, 
                v: torch.Tensor) -> torch.Tensor:
        # q [B, Lq, d]
        # k [B, Lk, d]
        # v [B, Lk, d]
        # q @ k.T [B, Lq, Lk]
        return self.softmax((q @ k.permute(0, 2, 1)) /self.scale) @ v # [B, Lq, d]

q = torch.rand((4, 8, 64))
k = torch.rand((4, 16, 64))
v = torch.rand((4, 16, 64))
att = ScaledDotProductAttention(64)
att(q, k, v).shape

torch.Size([4, 8, 64])

## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [104]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, 
                 d_k: int, d_v: int) -> None:
        super().__init__()
        assert d_model % num_heads == 0
        self.d_m = d_model
        self.d_k = d_k
        self.d_v = d_v
        self.h = num_heads
        self.W_o = torch.nn.Linear(num_heads * d_v, d_model, bias=False)
        self.W_q = torch.nn.Linear(d_model, num_heads * d_k, bias=False)
        self.W_k = torch.nn.Linear(d_model, num_heads * d_k, bias=False)
        self.W_v = torch.nn.Linear(d_model, num_heads * d_v, bias=False)
        self.softmax = nn.Softmax(dim=-1)
        self.register_buffer("scale", torch.tensor(d_k ** 0.5))

    def forward(
            self,
            q: torch.Tensor,        # [B, Lq, d_model]
            k: torch.Tensor,        # [B, Lk, d_model]
            v: torch.Tensor         # [B, Lk, d_model]
        )  ->  torch.Tensor:
        B, Lq, _ = q.shape
        _, Lk, _ = k.shape

        q = self.W_q(q) # [B, Lq, h * d_k]
        k = self.W_k(k) # [B, Lk, h * d_k]
        v = self.W_v(v) # [B, Lk, h * d_v]

        q = q.reshape(B, Lq, self.h, self.d_k).transpose(1, 2) # [B, h, Lq, d_k]
        k = k.reshape(B, Lk, self.h, self.d_k).transpose(1, 2) # [B, h, Lk, d_k]
        v = v.reshape(B, Lk, self.h, self.d_v).transpose(1, 2) # [B, h, Lk, d_v]

        att_all_heads = self.softmax((q @ k.transpose(-2, -1)) /self.scale) @ v # [B, h, Lq, d_v]
        att_all_heads = att_all_heads.transpose(1, 2).reshape(B, Lq, -1) # [B, Lq, h * d_v]
        return self.W_o(att_all_heads) # [B, Lq, d_model]


q = torch.rand((4, 8, 512))
k = torch.rand((4, 16, 512))
v = torch.rand((4, 16, 512))

mha = MultiHeadAttention(512, 8, 64, 64)
mha(q, k, v).shape

torch.Size([4, 8, 512])

## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [109]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, 
                 d_k: int, d_v: int, hid_dim_mlp:int,
                 dropout_p: float) -> None:
        super().__init__()
        self.ln1 = torch.nn.LayerNorm(d_model)
        self.ln2 = torch.nn.LayerNorm(d_model)
        self.mha = MultiHeadAttention(d_model, num_heads, d_k, d_v)
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(d_model, hid_dim_mlp),
            torch.nn.GELU(),
            torch.nn.Dropout(dropout_p),
            torch.nn.Linear(hid_dim_mlp, d_model),
            torch.nn.Dropout1d(dropout_p),
        )



    def forward(self, t_prev: torch.Tensor) -> torch.Tensor:
        # t_prev 
        t = self.ln1(t_prev)
        t_temp = self.mha(t, t, t) + t_prev
        t_temp = self.ln2(t_temp)
        return self.mlp(t_temp) + t_temp

t =  torch.rand((4, 16, 512))
enc_layer = TransformerEncoderLayer(512, 8, 64, 64, 256, 0.2)
enc_layer(t).shape

torch.Size([4, 16, 512])

## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [121]:
class MLPBlock(torch.nn.Module):
    def __init__(self, in_out_dim,  hid_dim:int) -> None:
        super().__init__()
        self.fc1 = torch.nn.Linear(in_out_dim, hid_dim, bias=False)
        self.fc2 = torch.nn.Linear(hid_dim, in_out_dim, bias=False)
        self.gelu = torch.nn.GELU()
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc2(self.gelu(self.fc1(x)))

class MLPMixerBlock(torch.nn.Module):
    def __init__(self, num_patches: int, hidden_dim: int,
                 tokens_mlp_dim: int, channels_mlp_dim: int):
        super().__init__()
        self.ln1 = torch.nn.LayerNorm(hidden_dim)
        self.ln2 = torch.nn.LayerNorm(hidden_dim)
        self.mlp1 = MLPBlock(num_patches, tokens_mlp_dim)
        self.mlp2 = MLPBlock(hidden_dim, channels_mlp_dim)
        
    def forward(self, t: torch.Tensor) -> torch.Tensor:
        # t [B, N, D] 
        t_temp = self.ln1(t).transpose(1, 2) # [B, D, N]
        t_temp = self.mlp1(t_temp) # [B, D, T_DIM]
        t_temp = t_temp.transpose(1, 2) + t # [B, N, D]
        return self.mlp2(self.ln2(t_temp)) + t_temp # [B, N, D]

t =  torch.rand((4, 16, 512))
mlp_mixer = MLPMixerBlock(num_patches=16, hidden_dim=512, tokens_mlp_dim=256, channels_mlp_dim=2048)
mlp_mixer(t).shape

torch.Size([4, 16, 512])

## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [122]:
# в статье вообще GELU!!!
class ConvMixer(torch.nn.Module):
    def __init__(self, ch_dim: int, kernel_size:int) -> None:
        super().__init__()
        self.dw_conv = torch.nn.Conv2d(ch_dim, ch_dim, kernel_size, 
                                       groups=ch_dim, bias=False, 
                                       padding=kernel_size//2)
        self.pw_conv = torch.nn.Conv2d(ch_dim, ch_dim, 1, 
                                       bias=False)
        self.bn1 = torch.nn.BatchNorm2d(ch_dim)
        self.bn2 = torch.nn.BatchNorm2d(ch_dim)
        self.relu = torch.nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        tmp_x = self.bn1(self.relu(self.dw_conv(x)))
        return self.bn2(self.relu(self.pw_conv(tmp_x + x)))
    
x = torch.rand(4, 16, 64, 64)    
conv_mixer = ConvMixer(16, 3)
conv_mixer(x).shape

torch.Size([4, 16, 64, 64])

## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

Ответ:

Потому что во всех трёх моделях главное - смешивать информацию между патчами (tokens) и между каналами.
Attention делает это адаптивно от содержимого, а Mixer/Conv - фиксированными/структурными операторами (локальность + патчи + сильное смешивание каналов).


Общая формула, связывающая MHA, MLPMixer, ConvMixer

Пусть $X \in \mathbb{R}^{N \times D}$.

$$
Y = X + A(X)\,X, \qquad Z = Y + \mathrm{ChannelMLP}(Y)
$$

Разница — в $A(X)$ (mix по токенам/пространству).

Multi-Head Attention
$$
A(X) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right), \qquad
Q = XW_Q,\quad K = XW_K,\quad V = XW_V
$$

$$
A(X)X \equiv A(X)V
$$

MLPMixer (token-mixing)
$$
A(X)X \equiv \left(W_2\,\phi\!\left(W_1\,X^\top\right)\right)^\top
$$

ConvMixer (depthwise spatial mixing)
$$
A(X)X \equiv \mathrm{DWConv}_k(X)
$$

Плюсы/минусы
Multi-Head Attention

Плюс: контент-зависимое глобальное взаимодействие (лучше дальние связи)

Минус: O(N^2) по памяти/времени

MLPMixer

Плюс: просто и быстро (в основном matmul), глобальное смешивание без softmax

Минус: часто фиксирует N (число патчей), менее адаптивен чем attention

ConvMixer

Плюс: сильный vision-bias (локальность), O(N), очень эффективно на изображениях

Минус: глобальные связи набираются через глубину (нет all-to-all как в attention)
